In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import sys
sys.path.append("..")

import gc
import time

import torch
from vllm import LLM

from sal.config import Config

from core import bon_search_v1

from utils.load_data import load_data_hf

In [2]:
base_dir = '/groups/chichengz/tnn/datasets/'


# dataset
ds_split = "test"
ds_dir = base_dir + "/prm800k/math_splits"

# models to benchmark
llm_dirs = [
    base_dir + "Llama3.2-1B-Instruct",
    base_dir + "Llama3.2-3B-Instruct",
    base_dir + "Qwen2.5-3B-Instruct",
    base_dir + "Qwen2.5-7B-Instruct",
]


In [3]:
# general params
config = Config()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048

config.n = 32
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

num_trials = 2
level = 4

llm_gpu_memory_utilization = 0.7

In [4]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)

num_questions = len(dataset)
num_questions = min(num_questions, 10)
batch_of_questions = [dataset[i]['problem'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

num_questions = 10


### Benchmark `best_of_n_v1` across models
Each model is loaded, timed, then unloaded before the next to avoid OOM.

In [5]:
results_summary = []

for llm_dir in llm_dirs:
    model_name = llm_dir.rstrip('/').split('/')[-1]
    print(f"\n=== {model_name} ===")

    llm_vllm = LLM(
        model=llm_dir,
        tensor_parallel_size=1,
        max_model_len=5000,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        enforce_eager=True,
        distributed_executor_backend=None,
        dtype="float16",
        seed=config.seed,
    )
    gc.collect()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info(0)
    print(f'GPU memory used: {(total - free) / (1024**3):.2f} GB')

    trial_times = []
    for trial_idx in range(num_trials):
        start_time = time.time()
        bon_search_v1.best_of_n_v1(batch_of_questions, config, llm_vllm, trial_idx)
        elapsed = time.time() - start_time
        trial_times.append(elapsed)
        print(f"  trial {trial_idx}: {elapsed / num_questions:.4f}s/question  {elapsed:.2f}s total")

    avg_time = sum(trial_times) / len(trial_times)
    results_summary.append((model_name, avg_time, avg_time / num_questions))

    del llm_vllm
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== Summary ===")
print(f"{'model':<40} {'avg s/trial':>12} {'avg s/question':>15}")
print("-" * 70)
for model_name, avg_trial, avg_q in results_summary:
    print(f"{model_name:<40} {avg_trial:>12.2f} {avg_q:>15.4f}")


=== Llama3.2-1B-Instruct ===
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
GPU memory used: 22.85 GB
  trial 0: 4.0582s/question  40.58s total
  trial 1: 3.9035s/question  39.03s total

=== Llama3.2-3B-Instruct ===


Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:04<00:04,  4.71s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:05<00:00,  2.40s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:05<00:00,  2.75s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


GPU memory used: 22.87 GB
  trial 0: 10.6665s/question  106.67s total
  trial 1: 10.0521s/question  100.52s total


[rank0]:[W514 04:20:46.199757744 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== Qwen2.5-3B-Instruct ===


...


=== Qwen2.5-7B-Instruct ===


Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:01<00:05,  1.98s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:03<00:03,  1.99s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:05<00:01,  2.00s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:07<00:00,  1.93s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:07<00:00,  1.95s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


GPU memory used: 22.47 GB
  trial 0: 12.3199s/question  123.20s total
  trial 1: 11.6715s/question  116.71s total


[rank0]:[W514 04:31:12.576535436 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== Summary ===
model                                     avg s/trial  avg s/question
----------------------------------------------------------------------
Llama3.2-1B-Instruct                            39.81          3.9808
Llama3.2-3B-Instruct                           103.59         10.3593
Qwen2.5-3B-Instruct                            164.30         16.4305
Qwen2.5-7B-Instruct                            119.96         11.9957
